In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio

from tqdm.auto import tqdm
from pathlib import Path

In [ ]:
src_dir = (
    Path("~").expanduser()
    / "OneDrive - Stichting Deltares/PhD/Egypt/04_Data/2026_data/"
)
excel_path = (
    Path("~").expanduser()
    / "OneDrive - Stichting Deltares/PhD/Egypt_ERF_data/data_correlation.xlsx"
)
print (excel_path)


In [ ]:
survey_df = pd.read_csv(src_dir.parent / "Result/ind_survey_data_gov_assigned.csv")
distribution_df = pd.read_csv(
    src_dir.parent / "Result/population_density_matrix.csv", index_col="NAME1_"
)
masking_df = pd.read_excel(
    src_dir.parent / "Result/correlation.xlsx", sheet_name="Sheet1"
)
excel_df = pd.read_excel(excel_path, sheet_name="command_unit")

In [ ]:
print(f"Average survey weight: {survey_df['pweight'].mean()}")
print(f"Std of survey weight: {survey_df['pweight'].std()}")

In [ ]:
survey_df['sempinc'] = survey_df['sempinc'].fillna(0)
survey_df['irrgwag'] = survey_df['irrgwag'].fillna(0)
survey_df['totwag'] = survey_df['totwag'].fillna(0)

survey_df['totwag'] = survey_df['totwag'] + survey_df['sempinc'] + survey_df['irrgwag'] * 30

In [ ]:
female_df = survey_df[survey_df['sex_label'] == 'Female']
male_df = survey_df[survey_df['sex_label'] == 'Male']

In [ ]:
governorates = survey_df[survey_df["NAME1_"].notna()]["NAME1_"].unique()

rng = np.random.default_rng(seed=42)
sample_df = survey_df.copy(deep=True)
sample_df["command_unit"] = None
command_units = distribution_df.columns

gov_indices = {
    gov: idx.to_numpy() for gov, idx in survey_df.groupby("NAME1_").groups.items()
}
female_gov_indices = {
    gov: idx.to_numpy() for gov, idx in female_df.groupby("NAME1_").groups.items()
}
male_gov_indices = {
    gov: idx.to_numpy() for gov, idx in male_df.groupby("NAME1_").groups.items()
}

for gov, idx in gov_indices.items():
    probs = distribution_df.loc[gov].to_numpy()

    assignments = rng.choice(
        command_units,
        size=len(idx),
        p=probs,
    )

    sample_df.loc[idx, "command_unit"] = assignments

sample_df.loc[sample_df["command_unit"] == "other", "command_unit"] = None

sample_df = sample_df[sample_df["command_unit"].notna()]
final_sample = sample_df.merge(
    excel_df, left_on="command_unit", right_on="area", suffixes=("", "_spatial")
)

to_correlate = masking_df.loc[masking_df["correl"] == 1, "variable"]
cols = [c for c in to_correlate if c in final_sample.columns]

In [ ]:
#female_df = male_df.copy()

In [ ]:
female_df = female_df[[col for col in cols + ["NAME1_", "pweight" ] if col in female_df.columns]]

In [ ]:
#process categorical data

#marital. 1=married 0=single
female_df["mart_d"] = female_df["mart_d"].between(
    200, 300, inclusive="both"
).astype(int)

#empstab 1=stable 0=not
female_df["empstab"] = np.where(
    female_df["empstab"] == 1,
    1,
    0
)

#migrant 1=yes 0=no
female_df["immigr"] = female_df["immigr"].fillna(3)
female_df["immigr"] = np.where(
    female_df["immigr"] == 3,
    1,
    0
)
#emps 1=paid 0=unpaid
female_df["emps"] = np.where(
    (female_df["emps"] >= 5) & (female_df["emps"] != 6),
    1,
    0
)


In [ ]:
female_df["water productivity"] =  female_df["producer_price"] / female_df["water_supply"]

In [ ]:
v = female_df.columns.difference(["NAME1_", "pweight"])
female_df[v] = female_df[v].apply(pd.to_numeric, errors="coerce")
female_df["pweight"] = pd.to_numeric(female_df["pweight"], errors="coerce")


female_df = female_df.groupby("NAME1_").apply(
    lambda g: g[v].apply(lambda x: np.average(x, weights=g["pweight"])))


In [ ]:
base_cols = set(cols + ["area", "arable_km2", "rural_population", "population"])
keywords = ["salinity", "yield", "hectares"]

excel_df = excel_df[
    [
        col for col in excel_df.columns
        if col in base_cols
        or any(word in col.lower() for word in keywords)
    ]
]
#remove production as it seems incorrect
excel_df = excel_df.loc[:, ~excel_df.columns.str.startswith(("salinity"))]
excel_df.head()

In [ ]:
REPO_ROOT = Path.cwd().parent

CROSSWALK_CSV = REPO_ROOT / "ERF_Data/Data" / "crosswalk" / "command_area_governorate_crosswalk.csv"
crosswalk_df = pd.read_csv(CROSSWALK_CSV)
crosswalk_df.head()


excel_df["_id"] = excel_df["area"].str.extract(r"(\d+)$")[0].astype("Int64")


In [ ]:

excel_df["_id"] = excel_df["area"].str.extract(r"(\d+)$")[0].astype("Int64")

excel_df = excel_df.merge(
    crosswalk_df[
        ["command_area_objectid", "gov_NAME1_", "pct_of_governorate_area", "pct_of_command_area_area", "command_area_name"]
    ],
    left_on="_id",
    right_on="command_area_objectid",
    how="left"
).drop(columns=["_id", "command_area_objectid"])


In [ ]:
group_col = "gov_NAME1_"
source_id = "command_area_name"  # original unit holding the total

total_cols = [
    col for col in excel_df.columns
    if any(
        word in col.lower()
        for word in [
            "corrected",
            "arable",
            "water_supply",
            "water_demand",
            "price"
        ]
    )
    and col.lower() != "arable_km2_scaled"
]
w = "pct_of_command_area_area"

# All other numeric variables receive a weighted average
avg_cols = [
    col
    for col in excel_df.select_dtypes(include="number").columns
    if col not in total_cols
    and col not in {
        w,
        "pct_of_command_area_area",
        "pct_of_governorate_area",
        source_id,
    }
]

# Weighted averages
weighted_avg = excel_df.groupby(group_col).apply(
    lambda g: g[avg_cols].apply(
        lambda x: np.average(x, weights=g[w])
    )
)

# Convert weights into shares within each original source area
weight_share = excel_df[w].div(
    excel_df.groupby(source_id)[w].transform("sum")
)

# Distribute each absolute total and then sum by governorate
distributed_totals = excel_df[total_cols].mul(weight_share, axis=0)

absolute_totals = distributed_totals.groupby(
    excel_df[group_col]
).sum(min_count=1)

# Combine both types of results
result_excel = (
    weighted_avg
    .join(absolute_totals, how="outer")
    .reset_index()
)

#result_excel.to_csv("result_excel.csv")

In [ ]:


col_normalize = [
    col for col in excel_df.columns
    if any(
        word in col.lower()
        for word in [
            "corrected",
            "hectares",
            "arable",
            "water_supply",
            "water_demand"
        ]
    )
    and col.lower() != "arable_km2_scaled"
]

result_excel[col_normalize] = result_excel[col_normalize].div(
    result_excel["arable_km2"],
    axis=0
)

result_excel["producer_price_pp"] = result_excel["producer_price"].div(
    result_excel["rural_population"],
    axis=0
)

result_excel["arable_km2_pp"] = result_excel["arable_km2"].div(
    result_excel["rural_population"],
    axis=0
)
result_excel.head()

In [ ]:
corelate_df = female_df.merge(
     result_excel,
    left_on="NAME1_",
    right_on="gov_NAME1_",
    how="left"
)

corelate_df = corelate_df.drop(
    columns=[
        col for col in corelate_df.columns
        if "hectare" in col.lower()
        or col in ["pweight", "numwrksc", "disabl", "sex", "emps", "arable_km2", "population", "rural_population", "rwi_mean", "n_other_employed", "producer_price"]
    ],
    errors="ignore"
)
corelate_df.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Pearson correlation for all numeric variables
corr = corelate_df .select_dtypes("number").corr(method="pearson")

plt.figure(figsize=(20, 16))
sns.heatmap(
    corr,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    cbar_kws={"label": "Pearson correlation"}
)

plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


corr.columns = corr.columns.str.replace(",", " ", regex=False)
#corr.to_csv("male.csv")

In [ ]:
corr.columns = corr.columns.str.replace(",", " ", regex=False)
corr.index = corr.index.str.replace(",", " ", regex=False)

# Convert correlation matrix to pairwise table
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)

corr_long = (
    corr.where(mask)
        .stack()
        .reset_index()
)

corr_long.columns = ["var1", "var2", "femalecorel"]

# Round correlation values
corr_long["femalecorel"] = corr_long["femalecorel"].round(2)

corr_long.to_csv("female.csv", index=False)

corr_long.head(20)